## Preprocessing of Midi Files

Goals:
- read MIDI files
- align with CSV file (train, test, validation split)
- filter non-4/4
- filter fills
- extract Information: 
    - onsets (hits) (binary)
    - continuous derivation (-.5, .5)
    - velocity (0 - 127 --> 0 - 1)

TODOS
- train, test, validation split
- remove duplicates if 2 hits on same instrument are quantized to same timestep --> use louder one


Problems:
- The original paper (2019) summarizes into 9 categories that differ from the second paper (where we derive our dataset), which has 7 categories; For now I use the first mapping with 9 instruments. 
- flattening the matrices with variable-length sequences?

In [2]:
import numpy as np
import pandas as pd
from pathlib import Path
import time

# Import preprocessing functions from module
from preprocessing import extract_matrices, N_INSTRUMENTS

# ! python -m pip install ipywidgets

In [3]:
# Constants

# DS_ROOT = "e-gmd-v1.0.0"
DS_ROOT = "../../e-gmd-v1.0.0" # this assumes the dataset to be in a solder next to the git repository
N_INSTRUMENTS = 9

In [4]:
# pitch_to_category function is now in preprocessing.py module

In [5]:
# read_midi function is now in preprocessing.py module

In [6]:
# extract_matrices function is now in preprocessing.py module

In [7]:
# Iterate over csv
# read all MIDI files

csv_path = Path(DS_ROOT) / "e-gmd-v1.0.0.csv"
df = pd.read_csv(csv_path)

# filter all non 4-4
df_filt = df[df['time_signature'] == '4-4']
filter_count = len(df) - len(df_filt)
print(f"Removed {filter_count} files due to mismatching time signature")

# split in train, test, validaton
df_train = df_filt[df_filt['split'] == 'train']
df_test = df_filt[df_filt['split'] == 'test']
df_validation = df_filt[df_filt['split'] == 'validation']

# Use parallel processing from preprocessing module
train_set = extract_matrices(df_train, DS_ROOT)
test_set = extract_matrices(df_test, DS_ROOT)
validation_set = extract_matrices(df_validation, DS_ROOT)

Removed 516 files due to mismatching time signature


  0%|          | 0/34787 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# Save datasets in efficient format for PyTorch
# Using .npz (compressed numpy) format for efficient storage and fast loading

import os

# Create output directory if it doesn't exist
output_dir = Path(DS_ROOT)
os.makedirs(output_dir, exist_ok=True)

print(f"Saving datasets to {output_dir}...")

# Save train set - use object array to handle variable lengths
train_path = output_dir / "train_set.npz"
train_array = np.empty(len(train_set), dtype=object)
for i, item in enumerate(train_set):
    train_array[i] = item
np.savez_compressed(train_path, data=train_array)
print(f"Saved train set: {len(train_set)} files -> {train_path}")

# Save test set
test_path = output_dir / "test_set.npz"
test_array = np.empty(len(test_set), dtype=object)
for i, item in enumerate(test_set):
    test_array[i] = item
np.savez_compressed(test_path, data=test_array)
print(f"Saved test set: {len(test_set)} files -> {test_path}")

# Save validation set
validation_path = output_dir / "validation_set.npz"
validation_array = np.empty(len(validation_set), dtype=object)
for i, item in enumerate(validation_set):
    validation_array[i] = item
np.savez_compressed(validation_path, data=validation_array)
print(f"Saved validation set: {len(validation_set)} files -> {validation_path}")

# Print file sizes
for path in [train_path, test_path, validation_path]:
    size_mb = path.stat().st_size / (1024 * 1024)
    print(f"{path.name}: {size_mb:.2f} MB")

print("\nDatasets saved successfully!")

NameError: name 'Path' is not defined

In [ ]:
# Free up memory after saving datasets
import gc

print("Freeing memory...")
initial_objects = len(gc.get_objects())

# Delete dataset variables
del train_set, test_set, validation_set

# Force garbage collection
gc.collect()

final_objects = len(gc.get_objects())
print(f"Datasets removed from memory")
print(f"Objects freed: {initial_objects - final_objects}")

In [ ]:
# Example: How to load the saved datasets for PyTorch training
# Uncomment and run this cell to test loading

# Load train set
# loaded_data = np.load(output_dir / "train_set.npz", allow_pickle=True)
# train_set_loaded = loaded_data['data']
# print(f"Loaded train set: {len(train_set_loaded)} samples")
# print(f"First sample shape: {train_set_loaded[0].shape}")

# Example PyTorch Dataset class:
"""
import torch
from torch.utils.data import Dataset

class DrumDataset(Dataset):
    def __init__(self, npz_path):
        data = np.load(npz_path, allow_pickle=True)
        self.data = data['data']
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        # Returns tensor of shape (3, 9, n_timesteps)
        # 3 channels: onset, offset, velocity
        # 9 instruments
        return torch.from_numpy(self.data[idx]).float()

# Usage:
# train_dataset = DrumDataset('../data/processed/train_set.npz')
# train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True)
"""

## Save Processed Datasets

Saving the preprocessed data for efficient loading in PyTorch training.

## Questions

- do i concatenate the 2-bar snippets (sliding window of 1 bar) fully, step-by-step or not at all?

In [ ]:
len(df)

45537